In [49]:
import gc
import warnings
from typing import Dict, Any, List

# --- Third‑party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scvi
import torch
import tensorflow as tf
#import anndata
from anndata import AnnData
from matplotlib.lines import Line2D
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from scvi.dataloaders import DataSplitter

# --- Project‑specific modules
# NOTE: keep sys.path adjustments *before* relative imports
sys.path.append(
    "/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/run_models/scVI"
)
sys.path.append(
    "/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD"
)

from model_config import *  # noqa: F403  (imports model_params_dict, build_model_dict, etc.)
from scMEDAL.utils.utils import (
    create_folder,
    read_adata,
    get_OHE,
    min_max_scaling,
    plot_rep,
    calculate_merge_scores,
    get_split_paths,
    calculate_zscores,
    get_clustering_scores_optimized,
)
from scMEDAL.utils.callbacks import ComputeLatentsCallback
from scMEDAL.utils.model_train_utils import ModelManager, load_data  #, compute_scores

# Silence potential tensorflow warnings for cleaner logs
warnings.filterwarnings("ignore", category=FutureWarning)
print("tf_version", tf.__version__)

import time
import glob



from paths_config import results_path_dict, input_base_path

from scMEDAL.utils.model_train_utils import (
    ModelManager,
    run_model_pipeline_LatentClassifier_v2,
    calculate_metrics_with_ci
)
from model_config import (
    data_path, model_params_dict, base_paths_dict, run_name,
    LatentClassifier_config, load_latent_spaces_dict,
    saved_models_base, source_file
)

from scMEDAL.utils.compare_results_utils import (
    get_input_paths_df,
    get_latent_paths_df,
    create_latent_dict_from_df
)




# --------------------------------------------------------------------------------------
# 1. Get Input Paths and Latent Paths
# --------------------------------------------------------------------------------------
df_latent = get_latent_paths_df(results_path_dict, k_folds=5)
df_inputs = get_input_paths_df(input_base_path, k_folds=5, eval_test=True)

# Merge latent and input paths
df = pd.merge(df_latent, df_inputs, on=["Split", "Type"], how="left")
print("Reading paths,\ndf paths:\n", df.head(5))

# --------------------------------------------------------------------------------------
# 2. Define Variables Necessary to Load Data and Train Model
# --------------------------------------------------------------------------------------
batch_col = model_params_dict['batch_col']
bio_col = model_params_dict['bio_col']

# Load metadata
metadata_all = pd.read_csv(glob.glob(os.path.join(data_path, "*meta.csv"))[0])
metadata_all[bio_col] = metadata_all[bio_col].astype('category')

# Define batch column (original metadata does not include batch yet)
metadata_all[batch_col] = metadata_all["batch"].astype('category')

print("n batches:", len(np.unique(metadata_all[batch_col]).tolist()))

# Define categories for batch and bio columns
batch_col_categories = np.unique(metadata_all[batch_col]).tolist()
print("check ordered batches:", batch_col_categories)

bio_col_categories = np.unique(metadata_all[bio_col]).tolist()
print("bio categories:", bio_col_categories)

# --------------------------------------------------------------------------------------
# 3. Update the Config for the Model
# --------------------------------------------------------------------------------------
load_latent_spaces_dict['batch_col_categories'] = batch_col_categories
load_latent_spaces_dict['bio_col_categories'] = bio_col_categories

# Update model
# LatentClassifier_config['Model'] = MixedEffectsModel

# Create latent path dictionary
latent_path_dict = create_latent_dict_from_df(df_latent)
load_latent_spaces_dict["latent_path_dict"] = latent_path_dict

tf_version 2.19.0
Reading paths,
df paths:
           Key  Split                                         LatentPath  \
0  scMEDAL-RE      1  /archive/bioinformatics/DLLab/AixaAndrade/src/...   
1  scMEDAL-RE      1  /archive/bioinformatics/DLLab/AixaAndrade/src/...   
2  scMEDAL-RE      1  /archive/bioinformatics/DLLab/AixaAndrade/src/...   
3  scMEDAL-RE      2  /archive/bioinformatics/DLLab/AixaAndrade/src/...   
4  scMEDAL-RE      2  /archive/bioinformatics/DLLab/AixaAndrade/src/...   

    Type                                         InputsPath  
0  train  /archive/bioinformatics/DLLab/AixaAndrade/src/...  
1    val  /archive/bioinformatics/DLLab/AixaAndrade/src/...  
2   test  /archive/bioinformatics/DLLab/AixaAndrade/src/...  
3  train  /archive/bioinformatics/DLLab/AixaAndrade/src/...  
4    val  /archive/bioinformatics/DLLab/AixaAndrade/src/...  
n batches: 31
check ordered batches: ['donor_1823', 'donor_4341', 'donor_4849', 'donor_4899', 'donor_5144', 'donor_5163', 'donor_5242

In [8]:
latent_path_dict.keys()

dict_keys(['scMEDAL-RE', 'AE', 'AEC', 'scMEDAL-FEC', 'scMEDAL-FE', 'scVI'])

In [5]:
np.unique(df_latent["Key"])

array(['AE', 'AEC', 'scMEDAL-FE', 'scMEDAL-FEC', 'scMEDAL-RE', 'scVI'],
      dtype=object)

In [6]:
# --------------------------------------------------------------------------------------
# 4. Run the Classifier for All Folds Latent Space
# --------------------------------------------------------------------------------------
folds_list = [1] #list(range(1, 6))  # Folds 1 to 5
all_folds_metrics_df = pd.DataFrame()

for fold in folds_list:
    print("fold", fold)
    load_latent_spaces_dict["fold"] = fold

    # Initialize model manager
    model_manager = ModelManager(
        model_params_dict,
        base_paths_dict,
        run_name,
        save_model=LatentClassifier_config["save_model"],
        use_kfolds=True,
        kfold=load_latent_spaces_dict["fold"]
    )

    # Update LatentClassifier config
    load_latent_spaces_dict["model_params"] = model_manager.params
    pipeline_LatentClassifier_config = {**load_latent_spaces_dict, **LatentClassifier_config}

fold 1
Created folder: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/saved_models/log_transformed_2916hvggenes/MEC/run_latent_classifier_scVI_fe_latent-scVI_latent_2025-05-22_11-33/splits_1
Created folder: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/figures/log_transformed_2916hvggenes/MEC/run_latent_classifier_scVI_fe_latent-scVI_latent_2025-05-22_11-33/splits_1
Created folder: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/latent_space/log_transformed_2916hvggenes/MEC/run_latent_classifier_scVI_fe_latent-scVI_latent_2025-05-22_11-33/splits_1
Folder already exists: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../outputs/ASD_outputs/saved_models/log_transformed_2916hvggenes/MEC/run_latent_classifier_scVI_fe_latent-scVI_latent_2025-05-2

In [15]:

pipeline_LatentClassifier_config['issparse']=False
pipeline_LatentClassifier_config['load_dense']=True

In [16]:
config = pipeline_LatentClassifier_config

# Load initial dataset paths and data
input_path_dict = get_split_paths(base_path=config['base_path'], fold=config['fold'])
adata_dict = load_data(
    input_path_dict,
    eval_test=config['model_params'].eval_test,
    scaling=config['model_params'].scaling,
    issparse=config['issparse'],
    load_dense=config['load_dense']
)

# Initialize dataset list and add 'test' dataset if evaluation on 'test' is enabled
dataset_list = ["train", "val"]
if config['model_params'].eval_test:
    dataset_list.append("test")

# Iterate through each model and dataset
for model in config['models_list']:
    for dataset in dataset_list:
        # Load the latent space for the current model and dataset

        latent_space_path = config['latent_path_dict'][model]["splits_" + str(config['fold'])][dataset]
        if isinstance(latent_space_path, (np.ndarray,list)):
            latent_space_path = latent_space_path[0]
            latent_space = np.load(latent_space_path)

            latent_key = f"{model}_latent_{dataset}"
            # save latent space in .obsm as latent_key
            adata_dict[dataset].obsm[latent_key] = latent_space
  


Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data/log_transformed_2916hvggenes/splits/split_1/train
X.shape before scaling (62735, 2916)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data/log_transformed_2916hvggenes/splits/split_1/val


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (20912, 2916)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data/log_transformed_2916hvggenes/splits/split_1/test


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (20912, 2916)


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [5]:
pipeline_LatentClassifier_config

{'latent_path_dict': None,
 'model_params': <scMEDAL.utils.model_train_utils.ModelManager.Namespace at 0x2aacf4493e60>,
 'base_path': '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/HealthyHeart/../data/HealthyHeart_data/log_transformed_3000hvggenes/splits',
 'fold': 1,
 'models_list': ['scMEDAL-RE', 'scVI'],
 'batch_col_categories': None,
 'bio_col_categories': None,
 'issparse': True,
 'load_dense': True,
 'batch_col': 'batch',
 'bio_col': 'celltype',
 'Model': None,
 'build_model_dict': {'n_latent_dims': 2,
  'layer_units': [8, 4],
  'n_pred': 13,
  'add_re_2_meclass': False,
  'name': 'mec'},
 'compile_dict': {'optimizer': <keras.src.optimizers.adam.Adam at 0x2aacf1cc8ec0>,
  'loss': <LossFunctionWrapper(<function categorical_crossentropy at 0x2aacf4538680>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>,
  'loss_weights': 1,
  'metrics': [<CategoricalAccuracy name=accuracy>]},
 'save_model': True,
 'latent_keys_config':

In [17]:
from scMEDAL.utils.model_train_utils import load_latent_spaces
config = pipeline_LatentClassifier_config

adata_dict = load_latent_spaces(
    base_path=config['base_path'],
    fold=config['fold'],
    models_list=config['models_list'],
    latent_path_dict=config['latent_path_dict'],
    model_params=config['model_params'],
    batch_col=config['batch_col'],
    bio_col=config['bio_col'],
    batch_col_categories=config['batch_col_categories'],
    bio_col_categories=config['bio_col_categories'],
    issparse=config['issparse'],
    load_dense=config['load_dense']
)


Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data/log_transformed_2916hvggenes/splits/split_1/train
X.shape before scaling (62735, 2916)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data/log_transformed_2916hvggenes/splits/split_1/val


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (20912, 2916)
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data/log_transformed_2916hvggenes/splits/split_1/test


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


X.shape before scaling (20912, 2916)


/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_scvi/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [18]:
adata_dict

{'train': AnnData object with n_obs × n_vars = 62735 × 2916
     obs: 'Unnamed: 0.1', 'Unnamed: 0', 'cell', 'cluster', 'sample', 'individual', 'region', 'age', 'sex', 'diagnosis', 'Capbatch', 'Seqbatch', 'post-mortem interval (hours)', 'RNA Integrity Number', 'genes', 'UMIs', 'RNA mitochondr. percent', 'RNA ribosomal percent', 'celltype', 'donor', 'batch', 'n_genes', 'original_index'
     var: 'Unnamed: 0'
     obsm: 'scMEDAL-RE_latent_train', 'scVI_latent_train',
 'val': AnnData object with n_obs × n_vars = 20912 × 2916
     obs: 'Unnamed: 0.1', 'Unnamed: 0', 'cell', 'cluster', 'sample', 'individual', 'region', 'age', 'sex', 'diagnosis', 'Capbatch', 'Seqbatch', 'post-mortem interval (hours)', 'RNA Integrity Number', 'genes', 'UMIs', 'RNA mitochondr. percent', 'RNA ribosomal percent', 'celltype', 'donor', 'batch', 'n_genes', 'original_index'
     var: 'Unnamed: 0'
     obsm: 'scMEDAL-RE_latent_val', 'scVI_latent_val',
 'test': AnnData object with n_obs × n_vars = 20912 × 2916
     obs:

In [19]:
adata_dict.keys()

dict_keys(['train', 'val', 'test', 'train_y', 'train_z', 'val_y', 'val_z', 'test_y', 'test_z'])

In [21]:
adata_dict['train_z']

,donor_1823,donor_4341,donor_4849,donor_4899,donor_5144,donor_5163,donor_5242,donor_5278,donor_5294,donor_5387,...,donor_5879,donor_5893,donor_5936,donor_5939,donor_5945,donor_5958,donor_5976,donor_5978,donor_6032,donor_6033
0,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62730,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
62731,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
62732,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
62733,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False


In [27]:
from scMEDAL.utils.model_train_utils import prepare_latent_space_inputs
print("Batches available: ", np.unique(adata_dict["train"].obs[batch_col]))

# 2. Prepare data for training
inputs = prepare_latent_space_inputs(adata_dict, config["latent_keys_config"], eval_test=config['model_params'].eval_test)

Batches available:  ['donor_1823' 'donor_4341' 'donor_4849' 'donor_4899' 'donor_5144'
 'donor_5163' 'donor_5242' 'donor_5278' 'donor_5294' 'donor_5387'
 'donor_5391' 'donor_5403' 'donor_5408' 'donor_5419' 'donor_5531'
 'donor_5538' 'donor_5554' 'donor_5565' 'donor_5577' 'donor_5841'
 'donor_5864' 'donor_5879' 'donor_5893' 'donor_5936' 'donor_5939'
 'donor_5945' 'donor_5958' 'donor_5976' 'donor_5978' 'donor_6032'
 'donor_6033']


In [37]:

rf_results = random_forest_accuracy_and_predictions(inputs, adata_dict,model_params=config['model_params'], eval_test=config['model_params'].eval_test)
rf_results

In [46]:
   # Evaluate using RandomForest
    
adata_dict = rf_results["adata_dict"]
rf_metrics = rf_results["metrics"]

# Calculate chance accuracy
chance_results = dummy_classifier_chance_accuracy(inputs, adata_dict, model_params=config['model_params'], eval_test=config['model_params'].eval_test,seed = config["seed"])
adata_dict = chance_results["adata_dict"]
chance_metrics = chance_results["metrics"]

# Merge the metrics from DFFN, SVM, and RandomForest
# metrics_df = pd.merge(dffn_metrics, svm_metrics, on="Dataset")
# metrics_df = pd.merge(metrics_df, rf_metrics, on="Dataset")
metrics_df = pd.merge(rf_metrics , chance_metrics, on="Dataset")

metrics_df.to_csv(os.path.join(config['model_params'].latent_path, "metrics.csv"))

In [47]:
metrics_df

,Dataset,RFAccuracy,RFBalancedAccuracy,ChanceAccuracy
0,train,0.999984,0.999979,0.080147
1,val,0.645180,0.564321,0.080337
2,test,0.641785,0.557263,0.078615


In [43]:
config["seed"]

42

In [30]:
inputs["train"].keys()

dict_keys(['fe_latent'])

In [25]:
config.keys()

dict_keys(['latent_path_dict', 'model_params', 'base_path', 'fold', 'models_list', 'batch_col_categories', 'bio_col_categories', 'batch_col', 'bio_col', 'Model', 'build_model_dict', 'compile_dict', 'save_model', 'latent_keys_config', 'return_metrics', 'return_adata_dict', 'return_trained_model', 'model_type', 'seed', 'issparse', 'load_dense'])

In [40]:



def random_forest_accuracy_and_predictions(inputs, adata_dict, model_params, eval_test=False):
    """
    Trains a RandomForest classifier using concatenated latent space features, evaluates it on train, validation, 
    and optionally the test datasets, and returns accuracy metrics along with updated AnnData objects 
    containing RandomForest predictions.

    Parameters:
    - inputs (dict): Dictionary containing input data for each dataset type ('train', 'val', 'test'). 
                     It includes keys 'fe_latent' and optionally 're_latent' for feature and regularization latent spaces.
    - adata_dict (dict): Dictionary containing AnnData objects or ground truth y values for 'train', 'val', and 'test' datasets.
    - model_params: Object containing model parameters, including the path to save predictions.
    - eval_test (bool): Boolean flag indicating whether to evaluate the model on the test set (default: False).

    Returns:
    - dict: A dictionary containing:
        - "metrics": DataFrame with accuracy and balanced accuracy metrics for the RandomForest classifier across train, validation, and optionally test datasets.
        - "adata_dict": Updated AnnData dictionary with true and predicted labels for each dataset, labeled with RandomForest predictions.
    """
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.ensemble import RandomForestClassifier

    def concatenate_features(dataset_type):
        if "re_latent" in inputs[dataset_type]:
            return np.concatenate((inputs[dataset_type]["fe_latent"], inputs[dataset_type]["re_latent"]), axis=1)
        else:
            return inputs[dataset_type]["fe_latent"]

    def process_dataset(dataset_type, X, y_true):
        # Make predictions
        y_pred = clf.predict(X)

        # Convert predictions to one-hot encoding
        y_pred_ohe = np.eye(num_classes)[y_pred]

        # Save true and predicted labels to adata_dict
        outputs_data = adata_dict[f'{dataset_type}_y']
        y_pred_df = pd.DataFrame(y_pred_ohe, columns=outputs_data.columns)
        adata_dict[dataset_type].obs["true_labels_rf"] = outputs_data.columns[outputs_data.values.argmax(axis=1)]
        adata_dict[dataset_type].obs["pred_labels_rf"] = y_pred_df.columns[y_pred_df.values.argmax(axis=1)]
        adata_dict[dataset_type].obs.to_csv(os.path.join(model_params.latent_path, f"y_pred_{dataset_type}_rf.csv"))

        # Calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)

        # Calculate balanced accuracy
        balanced_acc = balanced_accuracy_score(y_true, y_pred)

        # Return accuracy and balanced accuracy
        return {"accuracy": accuracy, "balanced_accuracy": balanced_acc, "adata_dict": adata_dict}

    # Initialize scaler and label encoder
    scaler = StandardScaler()
    label_encoder = LabelEncoder()

    # Prepare and standardize the training set
    X_train = concatenate_features("train")
    X_train = scaler.fit_transform(X_train)
    y_train = label_encoder.fit_transform(adata_dict["train_y"].values.argmax(axis=1))

    # Train the RandomForest classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    num_classes = adata_dict["train_y"].shape[1]

    # Standardize the validation set using the fitted scaler
    X_val = concatenate_features("val")
    X_val = scaler.transform(X_val)
    y_val = label_encoder.transform(adata_dict["val_y"].values.argmax(axis=1))

    # Process the train and validation sets
    train_results = process_dataset("train", X_train, y_train)
    val_results = process_dataset("val", X_val, y_val)

    # Initialize the metrics DataFrame
    metrics_df = pd.DataFrame({
        "Dataset": ["train", "val"],
        "RFAccuracy": [train_results["accuracy"], val_results["accuracy"]],
        "RFBalancedAccuracy": [train_results["balanced_accuracy"], val_results["balanced_accuracy"]]
    })

    # Evaluate on the test set if eval_test is True
    if eval_test:
        X_test = concatenate_features("test")
        X_test = scaler.transform(X_test)
        y_test = label_encoder.transform(adata_dict["test_y"].values.argmax(axis=1))
        test_results = process_dataset("test", X_test, y_test)

        try:
            # For pandas < 2.0
            metrics_df = metrics_df.append({
                "Dataset": "test",
                "RFAccuracy": test_results["accuracy"],
                "RFBalancedAccuracy": test_results["balanced_accuracy"]
            }, ignore_index=True)
        except AttributeError:
            # For pandas >= 2.0
            test_metrics = pd.DataFrame([{
                "Dataset": "test",
                "RFAccuracy": test_results["accuracy"],
                "RFBalancedAccuracy": test_results["balanced_accuracy"]
            }])
            metrics_df = pd.concat([metrics_df, test_metrics], ignore_index=True)


    adata_dict["train"] = train_results["adata_dict"]["train"]
    adata_dict["val"] = val_results["adata_dict"]["val"]

    return {"metrics": metrics_df, "adata_dict": adata_dict}


def dummy_classifier_chance_accuracy(inputs, adata_dict, model_params, eval_test=False,seed = 42):
    """
    Trains a DummyClassifier using the 'stratified' strategy to calculate the chance accuracy (baseline accuracy) 
    using concatenated latent space features. Evaluates it on train, validation, and optionally the test datasets, 
    and returns chance accuracy metrics.

    Parameters:
    - inputs (dict): Dictionary containing input data for each dataset type ('train', 'val', 'test'). 
                     It includes keys 'fe_latent' and optionally 're_latent' for feature and regularization latent spaces.
    - adata_dict (dict): Dictionary containing AnnData objects or ground truth y values for 'train', 'val', and 'test' datasets.
    - model_params: Object containing model parameters, including the path to save predictions.
    - eval_test (bool): Boolean flag indicating whether to evaluate the model on the test set (default: False).
    - seed (int) : seed set for repreducible results of dummy classifier with strategy: stratified

    Returns:
    - dict: A dictionary containing:
        - "metrics": DataFrame with chance accuracy and balanced accuracy metrics for the DummyClassifier across train, validation, and optionally test datasets.
        - "adata_dict": Updated AnnData dictionary with true and predicted labels for each dataset, labeled with DummyClassifier predictions.
    """
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.dummy import DummyClassifier

    def concatenate_features(dataset_type):
        if "re_latent" in inputs[dataset_type]:
            return np.concatenate((inputs[dataset_type]["fe_latent"], inputs[dataset_type]["re_latent"]), axis=1)
        else:
            return inputs[dataset_type]["fe_latent"]

    def process_dataset(dataset_type, X, y_true):
        # Make predictions
        y_pred = clf.predict(X)

        # Convert predictions to one-hot encoding
        y_pred_ohe = np.eye(num_classes)[y_pred]

        # Save true and predicted labels to adata_dict
        outputs_data = adata_dict[f'{dataset_type}_y']
        y_pred_df = pd.DataFrame(y_pred_ohe, columns=outputs_data.columns)
        adata_dict[dataset_type].obs["pred_labels_dummyclass"] = y_pred_df.columns[y_pred_df.values.argmax(axis=1)]
        adata_dict[dataset_type].obs.to_csv(os.path.join(model_params.latent_path, f"y_pred_{dataset_type}_dummy.csv"))

        # Calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)

        # Return accuracy and balanced accuracy
        return {"chance_accuracy": accuracy, "adata_dict": adata_dict}

    # Initialize scaler and label encoder
    scaler = StandardScaler()
    label_encoder = LabelEncoder()

    # Prepare and standardize the training set
    X_train = concatenate_features("train")
    X_train = scaler.fit_transform(X_train)
    y_train = label_encoder.fit_transform(adata_dict["train_y"].values.argmax(axis=1))

    # Train the DummyClassifier with the 'stratified' strategy
    clf = DummyClassifier(strategy="stratified",random_state = seed)
    clf.fit(X_train, y_train)
    num_classes = adata_dict["train_y"].shape[1]

    # Standardize the validation set using the fitted scaler
    X_val = concatenate_features("val")
    X_val = scaler.transform(X_val)
    y_val = label_encoder.transform(adata_dict["val_y"].values.argmax(axis=1))

    # Process the train and validation sets
    train_results = process_dataset("train", X_train, y_train)
    val_results = process_dataset("val", X_val, y_val)

    # Initialize the metrics DataFrame
    metrics_df = pd.DataFrame({
        "Dataset": ["train", "val"],
        "ChanceAccuracy": [train_results["chance_accuracy"], val_results["chance_accuracy"]],
    })

    # Evaluate on the test set if eval_test is True
    if eval_test:
        X_test = concatenate_features("test")
        X_test = scaler.transform(X_test)
        y_test = label_encoder.transform(adata_dict["test_y"].values.argmax(axis=1))
        test_results = process_dataset("test", X_test, y_test)

        try:
            # pandas < 2.0
            metrics_df = metrics_df.append({
                "Dataset": "test",
                "ChanceAccuracy": test_results["chance_accuracy"]
            }, ignore_index=True)
        except AttributeError:
            # pandas ? 2.0
            test_metrics = pd.DataFrame([{
                "Dataset": "test",
                "ChanceAccuracy": test_results["chance_accuracy"]
            }])
            metrics_df = pd.concat([metrics_df, test_metrics], ignore_index=True)

        adata_dict["test"] = test_results["adata_dict"]["test"]


    adata_dict["train"] = train_results["adata_dict"]["train"]
    adata_dict["val"] = val_results["adata_dict"]["val"]

    return {"metrics": metrics_df, "adata_dict": adata_dict}

In [11]:
config['models_list']

['scMEDAL-RE', 'scMEDAL-FE', 'scMEDAL-FEC', 'AEC', 'AE']

In [4]:

def run_model_pipeline_LatentClassifier_v2_PCA(Model, latent_path_dict, build_model_dict, compile_dict, model_params, save_model, 
                                           batch_col, bio_col, base_path, fold, models_list, latent_keys_config,
                                           batch_col_categories=None, bio_col_categories=None, return_metrics=True, 
                                           return_adata_dict=False, return_trained_model=False, model_type="mec",
                                           issparse=False, load_dense=False,seed=None):
    """
    Runs the complete model pipeline, including data loading, model training, evaluation, and metric collection.

    Parameters:
    - Model: The model class to be instantiated and trained.
    - latent_path_dict: Dictionary containing paths to latent space data for each model.
    - build_model_dict: Dictionary of parameters for building the model.
    - compile_dict: Dictionary of parameters for compiling the model.
    - model_params: Object containing additional model parameters and configurations.
    - save_model: Boolean flag indicating whether to save the trained model to disk.
    - batch_col: Name of the column representing batch information in the data.
    - bio_col: Name of the column representing biological information in the data.
    - base_path: Base directory path for datasets and model output.
    - fold: The specific fold identifier for cross-validation or data splitting.
    - models_list: List of models to be used in the pipeline.
    - latent_keys_config: Configuration dictionary for the latent keys used in model input.
    - batch_col_categories: List or array of categories for the batch column (optional).
    - bio_col_categories: List or array of categories for the biological column (optional).
    - return_metrics: Boolean flag indicating whether to return performance metrics (default: True).
    - return_adata_dict: Boolean flag indicating whether to return the AnnData dictionary (default: False).
    - return_trained_model: Boolean flag indicating whether to return the trained model (default: False).
    - model_type: String specifying the type of model being used (default: "mec").
    - issparse(bool): True if X is in sparse array, False if its dense
    - load_dense (bool): If True, forces conversion of sparse arrays to dense format.
    - seed (int) : seed set for repreducible results of dummy classifier with strategy: stratified. Default: None

    Returns:
    - results: Dictionary containing the results based on the specified flags. Possible keys include:
        - "dffn_model": The trained deep feedforward network model (if return_trained_model is True).
        - "metrics": DataFrame of performance metrics for the trained model and SVM.
        - "adata": The AnnData dictionary containing processed data (if return_adata_dict is True).
    """

    # 1. Load data latent paths and adata_dict
    adata_dict = load_latent_spaces(base_path, fold, models_list, latent_path_dict, model_params, batch_col, bio_col, batch_col_categories, bio_col_categories,issparse, load_dense)
    # Calculate PCA
    adata_dict = get_pca_andplot(adata_dict, plot_params=None, eval_test=model_params.eval_test,n_components=model_params.n_components,shape_color_dict = None)

    print("Batches available: ", np.unique(adata_dict["train"].obs[batch_col]))

    # 2. Prepare data for training
    inputs = prepare_latent_space_inputs(adata_dict, latent_keys_config, eval_test=model_params.eval_test)

    # 3. Build and train model, plott loss and evaluate dffn model
    dffn_results = build_train_evaluate_model(Model,build_model_dict, compile_dict, inputs, adata_dict, model_params, save_model, model_type)

    adata_dict = dffn_results["adata_dict"]
    dffn_metrics = dffn_results["metrics"]
    print("Training svc classifier")

    svm_results = svm_accuracy_and_predictions(inputs, adata_dict, model_params,eval_test=model_params.eval_test)
    adata_dict = svm_results["adata_dict"]
    svm_metrics = svm_results["metrics"]
    #metrics_df = pd.merge(dffn_metrics,svm_metrics)


    # Evaluate using RandomForest
    print("Training random forest classifier")

    rf_results = random_forest_accuracy_and_predictions(inputs, adata_dict, model_params, eval_test=model_params.eval_test)
    adata_dict = rf_results["adata_dict"]
    rf_metrics = rf_results["metrics"]

    # Calculate chance accuracy
    chance_results = dummy_classifier_chance_accuracy(inputs, adata_dict, model_params, eval_test=model_params.eval_test,seed=seed)
    adata_dict = chance_results["adata_dict"]
    chance_metrics = chance_results["metrics"]

    # Merge the metrics from DFFN, SVM, and RandomForest
    metrics_df = pd.merge(dffn_metrics, svm_metrics, on="Dataset")
    metrics_df = pd.merge(metrics_df, rf_metrics, on="Dataset")
    metrics_df = pd.merge(metrics_df, chance_metrics, on="Dataset")

    metrics_df.to_csv(os.path.join(model_params.latent_path, "metrics.csv"))

    


    # 7. Collect results based on flags
    results = {}
    if return_trained_model:
        results["dffn_model"] = dffn_results["model"]
    if return_metrics:
        results["metrics"] = metrics_df
    if return_adata_dict:
        results["adata"] = adata_dict

    return results

In [ ]:
# --------------------------------------------------------------------------------------
# 4. Run the Classifier for All Folds Latent Space
# --------------------------------------------------------------------------------------
folds_list = list(range(1, 6))  # Folds 1 to 5
all_folds_metrics_df = pd.DataFrame()

for fold in folds_list:
    print("fold", fold)
    load_latent_spaces_dict["fold"] = fold

    # Initialize model manager
    model_manager = ModelManager(
        model_params_dict,
        base_paths_dict,
        run_name,
        save_model=LatentClassifier_config["save_model"],
        use_kfolds=True,
        kfold=load_latent_spaces_dict["fold"]
    )

    # Update LatentClassifier config
    load_latent_spaces_dict["model_params"] = model_manager.params
    pipeline_LatentClassifier_config = {**load_latent_spaces_dict, **LatentClassifier_config}